<a href="https://colab.research.google.com/github/elliemci/agents/blob/main/agentic_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic RAG

## Required Libraries

In [1]:
!pip install datasets langchain langchain-community openai smolagents chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.5/114.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.4 MB/s eta 

In [ ]:
!pip install rich

## Imports

In [2]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/ColabNotebooks/AgentsCourse

Mounted at /content/drive
/content/drive/MyDrive/ColabNotebooks/AgentsCourse


In [3]:
from datasets import load_dataset
from langchain_community.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings #OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from smolagents import CodeAgent, Tool, HfApiModel

from langchain_community.embeddings import HuggingFaceEmbeddings

## Environment Varaibles

In [40]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["HUGGING_FACE_HUB_TOKEN"] = userdata.get('huggingface_hub_access_token')
os.environ["NEWS_API_KEY"] = userdata.get("NewsAPI_KEY")

## Tools

### Database Retrieval

1. Load MedMCQA

2. Prepare data: extract questions, contexts, and answers

3. Chunk the data

4. Embed and store in a vectorstore

5. Create a retriever tool

MedMCQA is a large-scale Multiple-Choice Question Answering dataset which contains real Medical exam Question Answering data set [medmcqa](https://huggingface.co/datasets/medmcqa) hast the follwing data fields:
* id: a string question identifier for each example
* question: a string text
* opa: Option A
* opb: Option B
* opc: Option D
* cop: Correct option
* choice_type {"single", "multi"}: question type, "single"-choice contains a single option, and "multi"-choice question contains a combination of multiple options
* exp: expert's explanation of the answer
* subject_name: medical subject name of thequestion
* topic_name: medical topic name

In [ ]:
# load MedMCQA from Huggingface
dataset = load_dataset("medmcqa", split="train") # for speed use "train[:1000]"

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/85.9M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/936k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

In [ ]:
dataset[1]

{'id': 'e3d3c4e1-4fb2-45e7-9f88-247cc8f373b3',
 'question': 'Which vitamin is supplied from only animal source:',
 'opa': 'Vitamin C',
 'opb': 'Vitamin B7',
 'opc': 'Vitamin B12',
 'opd': 'Vitamin D',
 'cop': 2,
 'choice_type': 'single',
 'exp': "Ans. (c) Vitamin B12 Ref: Harrison's 19th ed. P 640* Vitamin B12 (Cobalamin) is synthesized solely by microorganisms.* In humans, the only source for humans is food of animal origin, e.g., meat, fish, and dairy products.* Vegetables, fruits, and other foods of nonanimal origin doesn't contain Vitamin B12 .* Daily requirements of vitamin Bp is about 1-3 pg. Body stores are of the order of 2-3 mg, sufficient for 3-4 years if supplies are completely cut off.",
 'subject_name': 'Biochemistry',
 'topic_name': 'Vitamins and Minerals'}

### Clean up NA data

In [ ]:
# handling missing values by converting to pandas df and using isna()
import pandas as pd

df = pd.DataFrame(dataset)
df.isna().sum()

,0
id,0
question,0
opa,0
opb,0
opc,0
opd,0
cop,0
choice_type,0
exp,21953
subject_name,0


In [ ]:
# drop rows with missing values in columns exp and topic_name
df = df.dropna(subset=["exp"])
print(df.isna().sum())
df = df.dropna(subset=["topic_name"])
print(df.isna().sum())

id                  0
question            0
opa                 0
opb                 0
opc                 0
opd                 0
cop                 0
choice_type         0
exp                 0
subject_name        0
topic_name      73792
dtype: int64
id              0
question        0
opa             0
opb             0
opc             0
opd             0
cop             0
choice_type     0
exp             0
subject_name    0
topic_name      0
dtype: int64


In [ ]:
# convert no missin values dataframe back to datasest
dataset = dataset.from_pandas(df)

In [ ]:
# combine question and explanation for context
docs = []

for item in dataset:
  content = f"Q: {item['question']}\nA) {item['opa']} B) {item['opb']} C) {item['opc']} D) {item['opd']}\nAnswer: {item['cop']}\nExplanation: {item.get('exp', '')}"
  docs.append(Document(page_content=content, metadata={"id": item['id']}))

### Medical Dataset Retrieval Tool

In [ ]:
def get_correct_answer_text(ex):
    """
    Given an example from MedMCQA, extract the actual text(s) of the correct answer(s).
    Works for both 'single' and 'multi' choice_type.
    """
    options = {
        "1": ex.get("opa", "").strip(),
        "2": ex.get("opb", "").strip(),
        "3": ex.get("opc", "").strip(),
        "4": ex.get("opd", "").strip()
    }

    # get the text from the correct answer or answers
    correct_option_raw = str(ex.get("cop", "")).strip()
    correct_indices = [opt.strip() for opt in correct_option_raw.split(",") if opt.strip()]

    # Fallback: infer choice_type from number of correct answers
    declared_type = ex.get("choice_type", "single").strip().lower()
    inferred_type = "multi" if len(correct_indices) > 1 else "single"

    # Use whichever is more accurate
    choice_type = inferred_type if declared_type not in {"single", "multi"} else declared_type

    correct_texts = [options.get(idx, f"[Unknown Option {idx}]") for idx in correct_indices]

    return correct_texts if choice_type == "multi" else correct_texts[0]


In [ ]:
ex = {
    "question": "What causes increased blood pressure?",
    "opa": "Low salt intake",
    "opb": "Dehydration",
    "opc": "High sodium levels",
    "opd": "Regular exercise",
    "cop": "2, 3",
    "choice_type": "multi",
    "exp": "High sodium levels cause water retention, increasing blood volume and pressure.",
    "subject_name": "Physiology",
    "topic_name": "Cardiovascular System"
}

print(get_correct_answer_text(ex))

['Dehydration', 'High sodium levels']


In [5]:
import shutil
from tqdm import tqdm

# Setup with embeding-based retriever sentence-transformer
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = []

# Utility to get correct answer text
def get_correct_answer_text(ex):
    options = {
        "1": ex.get("opa", "").strip(),
        "2": ex.get("opb", "").strip(),
        "3": ex.get("opc", "").strip(),
        "4": ex.get("opd", "").strip()
    }

    correct_option_raw = str(ex.get("cop", "")).strip()
    correct_indices = [opt.strip() for opt in correct_option_raw.split(",") if opt.strip()]

    # fallback or check on consistency
    declared_type = ex.get("choice_type", "single").strip().lower()
    inferred_type = "multi" if len(correct_indices) > 1 else "single"
    choice_type = inferred_type if declared_type not in {"single", "multi"} else declared_type

    correct_texts = [options.get(idx, f"[Unknown Option {idx}]") for idx in correct_indices]
    return correct_texts if choice_type == "multi" else correct_texts[0]

# Tool definition
class MedDataChromaRetrieverTool(Tool):
    name = "med_data_chroma_retrieve"
    description = "Retrieves correct answers from MedMCQA using semantic similarity (Chroma vector store)."
    inputs = {
        "query": {
            "type": "string",
            "description": "The medical question you want to search."
        }
    }
    output_type = "string"

    def __init__(self, persist_directory="medmcqa_chromadb"):
        self.is_initialized = False
        self.vectorstore = None
        self.persist_directory = persist_directory

        if os.path.exists(persist_directory):
            self.vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding_model)
            self.is_initialized = True
        else:
            os.makedirs(self.persist_directory, exist_ok=True)
            self._build_vector_index()

    def _build_vector_index(self):
        med_dataset = load_dataset("medmcqa", split="train[:1000]")

        for ex in tqdm(med_dataset, desc="Building vector index for MedMCQA dataset"):
            correct_answer_text = get_correct_answer_text(ex)
            explanation = str(ex.get("exp", "") or "").strip()
            subject = str(ex.get("subject_name", "") or "").strip()
            topic = str(ex.get("topic_name", "") or "").strip()

            content = f"""Question: {ex['question']}
Correct Answer: {correct_answer_text}"""
            if explanation:
                content += f"\nExplanation: {explanation}"
            if subject and subject.upper() != "NA":
                content += f"\nSubject: {subject}"
            if topic and topic.upper() != "NA":
                content += f"\nTopic: {topic}"

            metadata = {
                "subject": subject,
                "topic": topic,
                "answer": correct_answer_text
            }

            split_docs = text_splitter.create_documents([content], metadatas=[metadata])
            docs.extend(split_docs)

        if os.path.exists(self.persist_directory):
            shutil.rmtree(self.persist_directory)

        self.vectorstore = Chroma.from_documents(
            docs,
            embedding=embedding_model,
            persist_directory=self.persist_directory
        )
        self.vectorstore.persist()
        self.is_initialized = True

    def forward(self, query: str):
        if not self.is_initialized:
            return "Vector store not initialized."

        results = self.vectorstore.similarity_search(query, k=3)
        return "\n\n".join([doc.page_content for doc in results])

# initialize the medical data retrieval tool
med_data_chroma_retrieve_tool = MedDataChromaRetrieverTool()


<ipython-input-5-de6afc2092f2>:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

<ipython-input-5-de6afc2092f2>:44: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  self.vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding_model)


### Add Conversation Memory

### Web Search

For **Hybrid RAG** and  Web Seach use dynamic Tool selection with ToolCallingAgent instead of CodeAgent, guide the LLM's behavior with tool descriptions or system prompts.

In [6]:
from smolagents import ToolCallingAgent, DuckDuckGoSearchTool

# Initialize model and tool
model = HfApiModel()

med_data_chroma_retrieve_tool.description = (
    "First try this tool. Retrieves accurate answers from the MedMCQA medical dataset. "
    "Use this for any standard medical knowledge, symptoms, treatments, or explanations."
)

web_tool = DuckDuckGoSearchTool()
web_tool.description = (
    "Use this only if the question cannot be answered from the medical dataset. "
    "Good for the latest or web-only info like new treatments, breaking news, or current guidelines."
)


### Hub Stats Tool

In [38]:
from smolagents import Tool
from huggingface_hub import list_models, model_info

from rich import print
from IPython.display import Markdown

class HubTopModelByTaskTool(Tool):
    """
    Fetches the most downloaded Hugging Face model used for a given task or in a field like finanial investments, medical, etc.
    """
    name = "hub_top_model_by_task"
    description = "Fetches the most downloaded Hugging Face model used for a given task or in a field like finanial investments, medical, etc."
    inputs = {
        "task": {
            "type": "string",
            "description": "The model name of a top performing model in a given field of interest."
        }
    }
    output_type = "string"

    def forward(self, task: str):
        try:
            # search and list models matching the task or keyword sorted by downloads
            models = list(list_models(search=task, sort="downloads", direction=-1, limit=1))

            if not models:
                return f"No models found for task '{task}'. Try something broader."

            top_model = models[0]
            info = model_info(top_model.id)

            description = info.cardData.get("summary") if info.cardData else None

            return (
                # emoji picker Ctrl + Cmd + Space
                f"🔥 Most Downloaded Model for [bold]'{task}':[/bold]\n"
                f"🔹 Model ID: [bold]{top_model.id}[/bold]\n"
                f"📥 Downloads: [bold]{top_model.downloads:,}[/bold]\n"
                f"📝 Description: {description or 'No description available.'}\n"
                f"🔗 View on Hugging Face: https://huggingface.co/{top_model.id}"
            )
        except Exception as e:
            return f"Error while searching models for '{task}': {str(e)}"

# Initialize the tool
hub_top_model_by_task_tool = HubTopModelByTaskTool()

# example usage: Get the most downloaded model used in medical research
print(hub_top_model_by_task_tool("medical research"))


🔥 Most Downloaded Model for 'medical research':
🔹 Model ID: AventIQ-AI/t5-summarization-for-medical-research-papers
📥 Downloads: 24
📝 Description: No description available.
🔗 View on Hugging Face: https://huggingface.co/AventIQ-AI/t5-summarization-for-medical-research-papers

### Latest News on a topic Tool

In [62]:
from smolagents import Tool
import os
import requests

class LatestNewsTool(Tool):
    name = "latest_news"
    description = "Fetches the latest news headlines on a specific topic."
    inputs = {
        "topic": {
            "type": "string",
            "description": "The topic or keywords for the latest news e.g., AI, economy, employment."
        }
    }
    output_type = "string"

    def __init__(self):
        super().__init__()
        self.api_key = os.environ.get("NEWS_API_KEY")
        self.base_url = "https://newsapi.org/v2/everything"
        self.is_initialized = True

        if not self.api_key:
            raise ValueError("NEWS_API_KEY environment variable not found. Set it before using this tool.")

    def forward(self, topic: str) -> str:
        try:
            print(f"Fetching news for topic: {topic}")

            params = {
                "q": topic,
                "apiKey": self.api_key,
                "language": "en",
                "sortBy": "publishedAt",
                "pageSize": 5
            }

            response = requests.get(self.base_url, params=params)
            response.raise_for_status()
            data = response.json()

            if not data.get("articles"):
                return f"No recent news found for topic: {topic}"

            results = []
            for article in data["articles"]:
                title = article.get("title", "No title")
                url = article.get("url", "")
                source = article.get("source", {}).get("name", "Unknown source")
                results.append(f"{title}\n{url} (Source: {source})")

            return "\n\n".join(results)

        except Exception as e:
            return f"Error fetching news: {str(e)}"



In [63]:
news_tool = LatestNewsTool()
print(news_tool("Humanoid robots"))

Fetching news for topic: Humanoid robots

A Review Of The Personal Humanoid Robots
https://slashdot.org/submission/17335387/a-review-of-the-personal-humanoid-robots (Source: Slashdot.org)

Links 4/20/2025
https://www.nakedcapitalism.com/2025/04/links-4-20-2025.html (Source: Nakedcapitalism.com)

Chinese Robots Can’t Outrun Human Marathoners Just Yet
https://uk.pcmag.com/ai/157654/chinese-robots-cant-outrun-human-marathoners-just-yet (Source: PCMag.com)

Chinese Robots Can’t Outrun Human Marathoners Just Yet
https://me.pcmag.com/en/ai/29441/chinese-robots-cant-outrun-human-marathoners-just-yet (Source: PCMag.com)

Watch: Humanoid robots compete against humans in half marathon
https://www.israelnationalnews.com/news/407082 (Source: Israelnationalnews.com)

In [65]:
model = HfApiModel()

agent = CodeAgent(
    tools=[med_data_chroma_retrieve_tool, web_tool, hub_top_model_by_task_tool],
    model=model
)

In [66]:
response = agent.run("What is the LLM most used in medical diagnostics?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the LLM most used in medical diagnostics?                                                               │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_result = web_search(query="most used LLM in medical diagnostics")                                         
  print(search_result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[The Open Medical-LLM Leaderboard: Benchmarking Large Language Models in 
...](https://huggingface.co/blog/leaderboard-medicalllm)
The Open Medical-LLM Leaderboard aims to address these challenges and limitations by providing a standardized 
platform for evaluating and comparing the performance of various large language models on a diverse range of 
medical tasks and datasets.

[A systematic review of large language model (LLM) evaluations in 
...](https://pmc.ncbi.nlm.nih.gov/articles/PMC11889796/)
Large Language Models (LLMs), advanced AI tools based on transformer architectures, demonstrate significant 
potential in clinical medicine by enhancing decision support, diagnostics, and medical education. However, their 
integration into clinical ...

[GitHub - AI-in-Health/MedLLMsPracticalGuide: [Nature Reviews 
...](https://github.com/AI-in-Health/MedLLMsPracticalGuide)
[Nature Reviews Bioengineering🔥] Application of Large Language Models in Medicine. A curated list of practical 
guide resources of Medical LLMs (Medical LLMs Tree, Tables, and Papers) - AI-in-Heal...

[LLMs in medicine: evaluations, advances, and the future](https://www.tanishq.ai/blog/posts/llm-medical-evals.html)
Specifically, most of the analyses discussed do not include processing of multiple modalities like medical images, 
lab tests, etc. which is an important aspect of clinical care. Instead, analyses of these tests are usually 
described in text and provided to the LLM. Additionally, much of the research focuses on evaluating general-purpose
LLMs.

[A generalist medical language model for disease diagnosis ... - 
Nature](https://www.nature.com/articles/s41591-024-03416-6)
Here we present MedFound, a generalist medical language model with 176 billion parameters, pre-trained on a 
large-scale corpus derived from diverse medical text and real-world clinical records.

[Large Language Models for Disease Diagnosis: A Scoping Review](https://arxiv.org/abs/2409.00097)
Despite the increasing attention in this field, a holistic view is still lacking. Many critical aspects remain 
unclear, such as the diseases and clinical data to which LLMs have been applied, the LLM techniques employed, and 
the evaluation methods used. In this article, we perform a comprehensive review of LLM-based methods for disease 
diagnosis.

[The application of large language models in medicine: A scoping 
review](https://www.sciencedirect.com/science/article/pii/S2589004224009350)
The surge in LLM-related research indicated a focus on medical writing, diagnostics, and patient communication, but
highlighted the need for careful integration, considering validation, ethical concerns, and the balance with 
traditional medical practice.

[Medical large language model for diagnostic reasoning across 
...](https://www.nature.com/articles/s41591-025-03520-1)
We sought to develop a medical LLM capable of 'understanding' diverse biomedical knowledge, while further 
fine-tuning its diagnostic assistance to learn physicians' inferential reasoning ...

[PDF](https://is.ijs.si/wp-content/uploads/2024/10/IS2024_-_CHATGPT_in_MEDICINE_paper_9-2.pdf)
One of the most critical aspects of benchmarking medical LLM's is comparing their performance with existing 
clinical decision support systems (CDSS) and other AI models. Traditional CDSS, often rule-based or statistical, 
have been used in healthcare for decades to assist clinicians in making evidence-based decisions.

[The Diagnostic Challenge: AI vs Doctors - 
marinpost.org](https://marinpost.org/blog/2024/12/26/the-diagnostic-challenge-ai-vs-doctors)
Diagnostic accuracy The short answer is that all LLMs performed poorly compared to doctors. Source: Hager [1] From 
the above visual data, we can extract the results comparing directly Llama 2 Chat (the most renowned of the general
open source LLM tested) vs doctors. And, Llama 2 Chat's diagnostic accuracy looks really poor compared to doctors. 
None of the other tested LLMs 

[Step 1: Duration 4.66 seconds| Input tokens: 2,185 | Output tokens: 63]

...

[Step 4: Duration 11.44 seconds| Input tokens: 15,247 | Output tokens: 432]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("GPT-4")                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: GPT-4

[Step 5: Duration 8.38 seconds| Input tokens: 22,447 | Output tokens: 582]